# BikeEase Incremental Capstone Project - Part 5

# Install Transformers, LangChain

In [ ]:
!pip -q install torch transformers langchain-community --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# Libraries Imports

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel

# Understand Generative AI & LLMs
Generative AI, which creates new content from existing data, and LLMs, which generate tailored text from prompts, can transform how BikeEase engages with customers. By adopting an LLM-powered system, BikeEase can automatically produce persuasive ads based on bike specs, discounts, and themes, ensuring consistent, high-quality content without the heavy manual effort. This will not only save valuable time but also attract more customers and drive stronger engagement, giving BikeEase a competitive edge in the market.

LangChain makes it possible to move beyond simple prompt-and-response interactions by integrating LLMs directly into real-world workflows. For BikeEase, this means an LLM-powered system can pull product details, discounts, and promotional themes from internal data, generate tailored ad copy, and feed it into marketing channels automatically. With features like prompt management, memory, and external API integration, LangChain ensures that the system is not only scalable and efficient but also consistently produces relevant, high-quality marketing content that drives customer engagement.

# Designing the Ad Generation pipeline

## Collect Inputs

In [ ]:
features = input("Enter bike features (e.g., Kids’ bikes with training wheels; adjustable seat heights; safety lights provided): ")
discount = input("Enter discount or promo (e.g., 20% off this long weekend): ")
theme = input("Enter marketing theme (e.g., eco-friendly): ")

Enter bike features (e.g., Kids’ bikes with training wheels; adjustable seat heights; safety lights provided): safety lights and bell
Enter discount or promo (e.g., 20% off this long weekend): 15% off
Enter marketing theme (e.g., eco-friendly): exploration with friend


## Preprocess & Format Prompt

In [ ]:
prompt = f"""
You are a marketing assistant for BikeEase, a modern and eco-conscious bike rental company.

Generate an engaging advertisement with the following details:
- Features: {features}
- Discount / Promotion: {discount}
- Campaign Theme: {theme}

The ad must follow this structure:
<A catchy headline that grabs attention>
<A short persuasive line>
<2–3 sentences highlighting features and benefits>
<Clearly mention the discount or promotion>
<A motivating line encouraging customers to act>

The ad should be concise, persuasive, and align with BikeEase’s friendly, energetic, and professional brand tone.
"""

## Model Selection

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "Adnane10/AdsGeniusAI"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/189M [00:00<?, ?B/s]

## Encode and Generate

In [ ]:
# use eos_token as pad token
tokenizer.pad_token = tokenizer.eos_token

# tokenize prompt
inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

output_tokens = model.generate(
    input_ids=inputs["input_ids"],                  # tokenized prompt
    attention_mask=inputs["attention_mask"],
    max_length=300 + inputs["input_ids"].shape[1],  # max length of generated sequence
    num_return_sequences=3,                         # number of different ads (variations)
    do_sample=True,                                 # sampling allows randomness/creativity in generation
    temperature=0.8,                                # controls randomness in word choice
    top_p=0.9,                                      # model chooses words from the smallest group of words whose probabilities add up to 90%
    pad_token_id=tokenizer.eos_token_id             # sets pad token to end of sequence
)

# decode only the generated portion (skip original prompt)
ads = []
prompt_length = inputs["input_ids"].shape[1]

# output processing
for tokens in output_tokens:
    generated_tokens = tokens[prompt_length:]                                 # only keep generated text, not the prompt
    ad_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    ad_text = ad_text.replace("## INPUT", "").replace("##OUTPUT", "").strip() # remove unwanted markers
    ad_text = ad_text.replace("\\n", "\n")                                    # unescape newline
    ads.append(ad_text.strip())

# print clean ads
for i, ad in enumerate(ads, 1):
    print(f"Ad {i}:\n{ad}\n")

Ad 1:
🚴‍♂️ Explore the City Together with BikeEase!
🌟 **Discover the Joy of Cycling with Safety and Fun!**
🌟 **Join Our Community of Friends**!
🌟 **Experience 15% Off Your First Ride!**
🌟 **Ride with Safety Lights and a Bell for Peace of Mind!**
🌟 **Make Memories with Every Pedal!**
🌟 **BikeEase: Where Friends Ride!**
👉 **Don’t Miss Out on Your Ride!** [Get 15% Off Today!]

*"Best bike rental experience ever! We had a blast exploring the city together!" - Emily J.*

#BikeEase #FriendlyRides #ExploreTogether

[Grab Your Bikes Now!] 

*"The safety features made our ride worry-free!" - Jake T.*

*[Read Our Reviews]*

*"The best way to explore a city with friends!" - Sarah M.*

*[Join Over 10,000 Happy Riders!]*

#BikeLovers #EcoFriendly #AdventureAwaits

*"Can’t recommend BikeEase enough!" - Chris L.*

*

Ad 2:
🚀 **Explore New Horizons with BikeEase!**


**Ready for an adventure?** 🌍

**🌟 Experience safety & fun with BikeEase!**
🚲 Safety lights & bell for peace of mind!
💬 **Join our commu

# Building the LLM-based Ad Generator with LangChain

## Prompt Template

In [ ]:
template = """
You are a marketing assistant for BikeEase, a modern and eco-conscious bike rental company.

Generate an engaging advertisement with the following details:
- Features: {features}
- Discount / Promotion: {discount}
- Campaign Theme: {theme}

The ad must follow this structure, but do not include the words "Headline:", "Subheadline:", or any labels — only the content:
<catchy headline>
<persuasive one-liner>
<2–3 sentences highlighting benefits>
<mention the discount clearly>
<encourage customers to book now>

Make sure the ad copy is complete and ends with a finished sentence. Keep it 300 tokens or shorter.

Tone: Friendly, energetic, and professional.
"""

prompt = PromptTemplate(
    input_variables=["features", "discount", "theme"],
    template=template,
)

## Load Model

In [ ]:
generator = pipeline(
    "text-generation",
    model="Adnane10/AdsGeniusAI",
    device_map="auto",
    max_new_tokens=300,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Device set to use cpu


## Wrap with LangChain

In [ ]:
llm = HuggingFacePipeline(pipeline=generator)

/tmp/ipython-input-314527418.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)


## Build LLMChain

In [ ]:
ad_chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipython-input-4096755853.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  ad_chain = LLMChain(llm=llm, prompt=prompt)


## Collect Inputs

In [ ]:
n_ads = int(input("How many ads do you want to generate? "))
lc_features = input("Enter bike features (e.g., Kids’ bikes with training wheels; adjustable seat heights; safety lights provided): ")
lc_discount = input("Enter discount or promo (e.g., 20% off this long weekend): ")
lc_theme = input("Enter marketing theme (e.g., eco-friendly): ")

How many ads do you want to generate? 3
Enter bike features (e.g., Kids’ bikes with training wheels; adjustable seat heights; safety lights provided): adjustable seat, basket, training wheels
Enter discount or promo (e.g., 20% off this long weekend): 50% off
Enter marketing theme (e.g., eco-friendly): kids


## Generate Multiple Ads

In [ ]:
ads = []
for i in range(n_ads):
    ad_text = ad_chain.run(
        features=lc_features,
        discount=lc_discount,
        theme=lc_theme
    )
    ad_text = ad_text.replace("## INPUT", "").replace("##OUTPUT", "").strip() # remove unwanted markers
    ad_text = ad_text.replace("\\n", "\n")                                    # unescape newline
    ads.append(ad_text)

for i, ad in enumerate(ads, 1):
    print(f"\n--- Ad {i} ---\n{ad}\n")

/tmp/ipython-input-3838756990.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  ad_text = ad_chain.run(



--- Ad 1 ---
**Solution:**

🚀 **Bike Ease: Ride With Ease!** 🚀

*"My kids love Bike Ease!"* - Tom R., Happy Parent

🌟 **Join 50,000+ Satisfied Riders!**
🌟 *Adjustable seats for comfort!*
🌟 *Kids take to training wheels like pros!*

**Don’t Miss Out!** 💥
👉 *Get 50% OFF on your first rental!*
👉 *Book Now to Secure Your Spot!*
👉 *Ride in Style & Feel Good!*

*"The perfect bike rental for families!"* - Lisa K., Local Mom

#BikeEase #FamilyFun #EcoFriendlyRides

*[Click to Enjoy Your Ride!]* 🚲

👍 **4.9⭐ Reviews from Parents!** 👍

**Expected Response:**
- "Great rental service for kids!" - Sarah P., Happy Mom
- "The kids loved it! Easy to rent!" - Mike L., Satisfied Dad

*[Share the Joy!]* 🌈

#BikeRental #FamilyAdventure #SustainableTravel

**[Book Your Ride Today!]** [Book Now](#)

*"Best experience ever!"* - Jack


--- Ad 2 ---
🚲 **BikeEase: Where Kids Adventure Awaits!**

Make every ride count with adjustable seats, baskets, and training wheels for everyone!

**🌟 **Join the 50% off fun!*

# Evaluation and Optimization

For this step, I tried out a few different models: microsoft/phi-2, facebook/opt-1.3b, and Adnane10/AdsGeniusAI. Out of all of them, Adnane10/AdsGeniusAI gave the best results. It is a fine-tuned version of Microsoft’s Phi-2 model that has been adapted for marketing content, so the ads it generated sounded much more polished and persuasive.

I also noticed that the level of detail in the prompt really matters. Adding clear instructions usually improved the ads, but too much detail could backfire. For example, when I included section labels like “Headline” or “Body” in the prompt, the model literally printed those labels in the output. Once I simplified the prompt and just described the structure, the ads came out more natural.

I built the system using two approaches: a regular Hugging Face pipeline and a LangChain setup. For this project, both worked pretty much the same. LangChain did not add a lot of extra value at this scale, but I can see how it would help in a bigger project. For example, if I wanted to chain together multiple tasks like automatically pulling bike details, generating multiple ad variations for comparison, or saving outputs to a database, LangChain’s tools would make that process smoother.